# 02 - Data Quality

**Objectif :** appliquer les règles qualité et les diagnostics colonnes uniquement sur le train brut.

In [1]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

Project root: /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab
Environment: development


## 1. Load raw train data

In [2]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

loader = CsvLoanDataLoader(path=settings.raw_train_path)
raw_train_df = loader.load()
raw_train_df.shape

2026-09-07 07:30:24 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/3-Mlops-Databricks-Projects/Clean-Architecture-MLops/Credit-Risk-Lab/data/raw/train.csv
2026-09-07 07:30:24 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (40500 lignes, 14 colonnes)


(40500, 14)

## 2. Quality checker

In [3]:
from credit_risk_lab.domain.entities import LoanSchema
from credit_risk_lab.infrastructure import CreditRiskQualityChecker

quality_checker = CreditRiskQualityChecker(schema=LoanSchema())
quality_report = quality_checker.validate(raw_train_df)
quality_report

QualityReport(rows=40500, columns=14, duplicate_rows=0, missing_values=0, invalid_age_rows=7, invalid_experience_rows=7)

## 3. Clean implausible rows

In [4]:
clean_train_df = quality_checker.clean(raw_train_df)

pd.DataFrame(
    [
        {"dataset": "raw_train", "rows": len(raw_train_df), "duplicates": raw_train_df.duplicated().sum()},
        {"dataset": "clean_train", "rows": len(clean_train_df), "duplicates": clean_train_df.duplicated().sum()},
    ]
)

,dataset,rows,duplicates
0,raw_train,40500,0
1,clean_train,40493,0


## 4. Inspect clean data

In [5]:
from credit_risk_lab.infrastructure.analytics import DatasetInspector

clean_inspector = DatasetInspector(clean_train_df)
display(clean_inspector.summary())
clean_inspector.target_distribution(settings.target_column)

DatasetSummary(rows=40493, columns=14, duplicate_rows=0, memory_mb=13.4685)

,class,rows,rate
0,0,31493,0.777739
1,1,9000,0.222261


## 5. Numeric outlier diagnostics

In [6]:
from credit_risk_lab.infrastructure.analytics import ColumnDiagnostics

diagnostics = ColumnDiagnostics(clean_train_df, target_column=settings.target_column)
numeric_outliers = diagnostics.numeric_outlier_report()
numeric_outliers

,feature,mean,median,std,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_rate
0,loan_amnt,9589.791445,8000.00,6304.908349,5000.00,12250.00,7250.00,-5875.000,23125.000,2105,0.051984
1,person_age,27.743141,26.00,5.902931,24.00,30.00,6.00,15.000,39.000,1951,0.048181
2,person_income,79977.010841,67040.00,63526.292895,47235.00,95960.00,48725.00,-25852.500,169047.500,1916,0.047317
3,person_emp_exp,5.387425,4.00,5.925382,1.00,8.00,7.00,-9.500,18.500,1532,0.037834
4,cb_person_cred_hist_length,5.862149,4.00,3.873317,3.00,8.00,5.00,-4.500,15.500,1224,0.030227
5,loan_percent_income,0.139821,0.12,0.087198,0.07,0.19,0.12,-0.110,0.370,671,0.016571
6,credit_score,632.607883,640.00,50.343070,602.00,670.00,68.00,500.000,772.000,456,0.011261
7,loan_int_rate,11.012309,11.01,2.980314,8.59,13.02,4.43,1.945,19.665,110,0.002717


## 6. Numeric outlier visuals

In [7]:
from credit_risk_lab.infrastructure.visualization import DataQualityVisualizer

visualizer = DataQualityVisualizer(
    frame=clean_train_df,
    target_column=settings.target_column,
)

visualizer.numeric_outlier_overview(
    numeric_outliers["feature"].head(6).tolist(),
).show()

## 7. Categorical diagnostics

In [8]:
categorical_profile = diagnostics.categorical_profile(rare_threshold=0.01)
categorical_profile

,feature,cardinality,missing_count,top_category,top_count,top_rate,rare_category_count,rare_categories
0,loan_intent,6,0,EDUCATION,8232,0.203294,0,[]
1,person_education,5,0,Bachelor,12063,0.297903,0,[]
2,person_home_ownership,4,0,RENT,21114,0.521423,1,[OTHER]
3,person_gender,2,0,male,22337,0.551626,0,[]
4,previous_loan_defaults_on_file,2,0,Yes,20572,0.508038,0,[]


## 8. Categorical visuals

In [9]:
visualizer.categorical_feature_overview(
    categorical_profile["feature"].head(4).tolist(),
).show()